In [6]:
# =============================================================================
# Zelle 01 – Setup & Daten laden (Hyperparameter-Tuning Modell B)
# =============================================================================
import sys
sys.path.append('../src')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from viz_config_v2 import apply_store44_style, save_figure, COLOR_GOLD, COLOR_BLUE, COLOR_GREEN, COLOR_TEXT_MUTED, BOXPLOT_STYLE
from preprocessing import load_dataset_b, FEATURE_SETS_B, WandstaerkeDNResidualizer, baue_preprocessing_pipeline_b, X_B_MERKMALE_ENCODED, Y_B_MERKMALE

SEED = 42
apply_store44_style()

df_b = load_dataset_b("../data/processed/model_b_preprocessed.csv")

# --- Identische Reproduktion Train/Test-Split und aeussere Folds (Notebook 11) ---
from sklearn.model_selection import train_test_split, KFold

train_idx, test_idx = train_test_split(df_b.index, test_size=0.2, random_state=SEED)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
fold_splits = list(kf.split(train_idx))
fold_splits_idx = [(train_idx[tr_pos], train_idx[val_pos]) for tr_pos, val_pos in fold_splits]

print(f"Datensatz: {df_b.shape}")
print(f"Train: {len(train_idx)}, Test: {len(test_idx)}, Folds: {len(fold_splits_idx)}")
df_b.head()

Datensatz: (2000, 14)
Train: 1600, Test: 400, Folds: 5


,material_mfr,dn_ziel,wandstaerke_soll,dickentoleranz,produktionsgeschwindigkeit_soll,ovalitaet_anforderung,schneckendrehzahl,massetemperatur,duesenspalt,vakuumniveau,innenluftdruck,kuehlwassertemperatur,schmelzfestigkeit_proxy,wandtyp_einwandig
0,0.760943,104.164565,1.385443,0.055494,9.400096,0.441754,33.691143,200.906186,1.434323,90.219856,78.197271,19.783884,0.539057,True
1,0.492003,158.608859,2.350657,0.295226,10.168837,0.353945,43.847410,199.855728,2.346622,104.351909,62.177159,19.459041,0.807997,False
2,0.850090,123.340827,1.836205,0.114787,10.907655,0.784632,45.499866,210.933172,1.929061,88.222130,105.947346,16.953516,0.449910,True
3,0.888113,225.335978,2.589429,0.198277,6.963402,0.734473,46.943818,208.291018,2.655661,104.679660,57.771840,21.110659,0.411887,True
4,0.309793,72.773106,1.553914,0.125063,12.279022,0.882630,36.970557,190.000000,1.597350,89.855192,91.287064,19.647650,0.990207,True


## Neue Hyperparameter-Suchräume für Notebook 12

Nach dem Hyperparameter-Herkunfts-Audit (Notebook 11, Abschluss) wurde
festgestellt, dass die meisten Modelle entweder unkritische sklearn-
Defaults nutzten oder Konfigurationen von Modell A (n=560 Training)
übernahmen, ohne für Modell B (n=1600 Training) geprüft zu sein.

**Begründung je Modell:**
- **Ridge:** `alpha` nie geprüft (nur sklearn-Default 1.0) → Suchraum
  [0.01, 0.1, 1.0, 10.0, 100.0] deckt schwache bis starke Regularisierung ab
- **kNN:** `n_neighbors=5` willkürlich übernommen; bei n=1600 (fast 3×
  mehr als Modell A) könnten mehr Nachbarn stabilere Vorhersagen liefern
  → Suchraum [3, 5, 10, 15, 25]
- **RandomForest:** bereits einmal punktuell korrigiert (Notebook 11,
  `max_depth=6`), aber nie systematisch geprüft, ob diese Zahl optimal
  ist → Suchraum `max_depth` [4, 6, 8, 12] (um den bisherigen Wert
  herum, mit Spielraum nach oben), `min_samples_leaf` [3, 5, 10]
- **MLP:** `hidden_layer_sizes=(50,)` nie hinterfragt (nur eine Schicht,
  50 Neuronen) → Suchraum [(50,), (100,), (50,50)] prüft groessere und
  tiefere Netze; zusätzlich `alpha` (L2-Regularisierung) [0.0001, 0.001,
  0.01], da bisher nicht reguliert
- **SVR:** `C=1.0, gamma="scale"` nie geprüft → Suchraum `C` [0.1, 1.0,
  10.0, 100.0], `gamma` ["scale", "auto", 0.01, 0.1]
- **HistGradientBoosting, XGBoost, LightGBM:** von Modell A (n=560)
  übernommene, stark regulierte Werte (`max_iter=50, max_depth=3` bzw.
  Ã¤quivalent) - bei n=1600 potenziell zu restriktiv → Suchraum
  `n_estimators`/`max_iter` [50, 100, 150, 200] (Bereich erweitert nach
  oben), `max_depth` [3, 5, 7] bzw. `num_leaves` [7, 15, 31] bei LightGBM

Ziel: fuer jedes Modell pruefen, ob die bisherige, teils uebernommene
Konfiguration tatsaechlich die beste war, oder ob systematisch bessere
Werte existieren - datengetrieben, nicht durch weitere Annahmen ersetzt.

In [7]:
# =============================================================================
# Zelle 02 – Modell-Registry mit Hyperparameter-Suchraeumen (alle 8 Modelle)
# =============================================================================
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

SUCHRAEUME_B = {
    "Ridge": {
        "modell": Ridge(random_state=SEED),
        "params": {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
        "struktur": "multioutput",
    },
    "kNN": {
        "modell": KNeighborsRegressor(),
        "params": {"n_neighbors": [3, 5, 10, 15, 25]},
        "struktur": "multioutput",
    },
    "RandomForest": {
        "modell": RandomForestRegressor(n_estimators=200, random_state=SEED),
        "params": {"max_depth": [4, 6, 8, 12], "min_samples_leaf": [3, 5, 10]},
        "struktur": "einzeln",
    },
    "MLP": {
        "modell": MLPRegressor(max_iter=2000, early_stopping=True, learning_rate_init=0.005, n_iter_no_change=20, random_state=SEED),
        "params": {"hidden_layer_sizes": [(50,), (100,), (50,50)], "alpha": [0.0001, 0.001, 0.01]},
        "struktur": "multioutput",
    },
    "SVR": {
        "modell": SVR(kernel="rbf"),
        "params": {"C": [0.1, 1.0, 10.0, 100.0], "gamma": ["scale", "auto", 0.01, 0.1]},
        "struktur": "einzeln",
    },
    "HistGradientBoosting": {
        "modell": HistGradientBoostingRegressor(random_state=SEED),
        "params": {"max_iter": [50, 100, 150, 200], "max_depth": [3, 5, 7]},
        "struktur": "einzeln",
    },
    "XGBoost": {
        "modell": XGBRegressor(random_state=SEED),
        "params": {"n_estimators": [50, 100, 150, 200], "max_depth": [3, 5, 7]},
        "struktur": "einzeln",
    },
    "LightGBM": {
        "modell": LGBMRegressor(random_state=SEED, verbose=-1),
        "params": {"n_estimators": [50, 100, 150, 200], "num_leaves": [7, 15, 31]},
        "struktur": "einzeln",
    },
}

for name, konfig in SUCHRAEUME_B.items():
    n_combos = int(np.prod([len(v) for v in konfig["params"].values()]))
    print(f"{name:22s} Struktur={konfig['struktur']:11s} {n_combos:3d} Kombinationen")

Ridge                  Struktur=multioutput   5 Kombinationen
kNN                    Struktur=multioutput   5 Kombinationen
RandomForest           Struktur=einzeln      12 Kombinationen
MLP                    Struktur=multioutput   9 Kombinationen
SVR                    Struktur=einzeln      16 Kombinationen
HistGradientBoosting   Struktur=einzeln      12 Kombinationen
XGBoost                Struktur=einzeln      12 Kombinationen
LightGBM               Struktur=einzeln      12 Kombinationen


In [8]:
# =============================================================================
# Zelle 03 – Pilot-Zeitschaetzung vor der vollstaendigen Nested-CV-Schleife
# =============================================================================
from sklearn.base import clone

AEUSSERE_FOLDS, INNERE_FOLDS = 5, 5

def geschaetzte_fits(n_combos, struktur):
    y_anzahl = 6 if struktur == "einzeln" else 1
    return (AEUSSERE_FOLDS * INNERE_FOLDS * n_combos + AEUSSERE_FOLDS) * y_anzahl

pilot_ergebnisse = []
prep = baue_preprocessing_pipeline_b("original")
X_probe = prep.fit_transform(df_b.loc[fold_splits_idx[0][0]])
y_probe_alle = df_b.loc[fold_splits_idx[0][0], Y_B_MERKMALE]

for modell_name, konfig in SUCHRAEUME_B.items():
    t0 = time.time()
    modell_probe = clone(konfig["modell"])
    if konfig["struktur"] == "multioutput":
        modell_probe.fit(X_probe, y_probe_alle)
    else:
        modell_probe.fit(X_probe, y_probe_alle.iloc[:, 0])
    einzel_fit_zeit = time.time() - t0

    n_combos = int(np.prod([len(v) for v in konfig["params"].values()]))
    gesamt_fits = geschaetzte_fits(n_combos, konfig["struktur"])
    geschaetzte_gesamtzeit = einzel_fit_zeit * gesamt_fits

    pilot_ergebnisse.append({"modell": modell_name, "n_combos": n_combos, "einzel_fit_sek": round(einzel_fit_zeit,4), "geschaetzte_gesamtzeit_sek": round(geschaetzte_gesamtzeit,1)})

pilot_df = pd.DataFrame(pilot_ergebnisse)
print(pilot_df.to_string(index=False))
print(f"\nGeschaetzte Gesamtzeit: {pilot_df['geschaetzte_gesamtzeit_sek'].sum():.0f}s ({pilot_df['geschaetzte_gesamtzeit_sek'].sum()/60:.1f} Minuten)")

              modell  n_combos  einzel_fit_sek  geschaetzte_gesamtzeit_sek
               Ridge         5          0.0038                         0.5
                 kNN         5          0.0000                         0.0
        RandomForest        12          0.8862                      1621.8
                 MLP         9          1.7162                       394.7
                 SVR        16          0.0460                       111.8
HistGradientBoosting        12          5.0243                      9194.4
             XGBoost        12          0.1199                       219.5
            LightGBM        12          0.0766                       140.2

Geschaetzte Gesamtzeit: 11683s (194.7 Minuten)


In [9]:
# =============================================================================
# Zelle 03b – HistGradientBoosting: Zeitmessung wiederholen (Reproduzierbarkeit)
# =============================================================================
from sklearn.ensemble import HistGradientBoostingRegressor

zeiten_wiederholt = []
for i in range(3):
    t0 = time.time()
    modell_probe = HistGradientBoostingRegressor(random_state=SEED)
    modell_probe.fit(X_probe, y_probe_alle.iloc[:, 0])
    zeiten_wiederholt.append(time.time() - t0)

print(f"Wiederholte Fit-Zeiten: {[round(z,3) for z in zeiten_wiederholt]}")

Wiederholte Fit-Zeiten: [0.294, 0.24, 0.217]


In [10]:
# =============================================================================
# Zelle 03c – RandomForest: Zeitmessung wiederholen (Reproduzierbarkeits-Check)
# =============================================================================
from sklearn.ensemble import RandomForestRegressor

zeiten_wiederholt_rf = []
for i in range(3):
    t0 = time.time()
    modell_probe = RandomForestRegressor(n_estimators=200, random_state=SEED)
    modell_probe.fit(X_probe, y_probe_alle.iloc[:, 0])
    zeiten_wiederholt_rf.append(time.time() - t0)

print(f"Wiederholte RandomForest Fit-Zeiten: {[round(z,3) for z in zeiten_wiederholt_rf]}")

Wiederholte RandomForest Fit-Zeiten: [0.82, 0.775, 0.774]


In [ ]:
# =============================================================================
# Zelle 04 (FINAL, korrigiert) – Nested-CV-Tuning: vollstaendiges Metrik-Set
# in einem einzigen konsistenten Durchlauf
# =============================================================================
# Korrekturen ggue. vorheriger Version: (1) Speicherort-Hinweis im Output,
# (2) try/except um das Speichern selbst (Lehre aus Notebook 11 - stiller
# Fehlschlag), (3) predict_zeit_sekunden_mean ergaenzt (Konsistenz mit
# Notebook 11 Metrik-Set).
# =============================================================================
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore")

def berechne_metriken_b(y_true, y_pred, y_true_std):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    return {
        "mae": mae, "rmse": rmse, "r2": r2, "median_ae": medae,
        "rmse_normiert": rmse / y_true_std if y_true_std > 0 else np.nan,
        "mae_normiert": mae / y_true_std if y_true_std > 0 else np.nan,
    }

y_std_je_merkmal = {y: df_b.loc[train_idx, y].std() for y in Y_B_MERKMALE}
OUTPUT_CSV_TUNING_B = "../reports/tables/12_tuning_results_model_b.csv"
tuning_ergebnisse_b = []
start_gesamt = time.time()

for modell_idx, (modell_name, konfig) in enumerate(SUCHRAEUME_B.items()):
    eintrag = {"modell": modell_name, "struktur": konfig["struktur"], "status": "ok"}
    fold_val_metriken, fold_train_metriken = [], []
    fold_best_params, fold_fit_zeiten, fold_predict_zeiten = [], [], []

    try:
        for tr_idx, val_idx in fold_splits_idx:
            prep = baue_preprocessing_pipeline_b("original")
            X_tr = prep.fit_transform(df_b.loc[tr_idx])
            X_val = prep.transform(df_b.loc[val_idx])
            y_tr = df_b.loc[tr_idx, Y_B_MERKMALE]
            y_val = df_b.loc[val_idx, Y_B_MERKMALE]
            innere_cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

            t0 = time.time()
            if konfig["struktur"] == "multioutput":
                grid = GridSearchCV(clone(konfig["modell"]), konfig["params"], scoring="r2", cv=innere_cv, n_jobs=-1)
                grid.fit(X_tr, y_tr)
                fit_zeit = time.time() - t0
                t1 = time.time()
                y_pred_val = pd.DataFrame(grid.predict(X_val), columns=Y_B_MERKMALE, index=val_idx)
                y_pred_train = pd.DataFrame(grid.predict(X_tr), columns=Y_B_MERKMALE, index=tr_idx)
                predict_zeit = time.time() - t1
                fold_best_params.append(grid.best_params_)
            else:
                y_pred_val = pd.DataFrame(index=val_idx, columns=Y_B_MERKMALE, dtype=float)
                y_pred_train = pd.DataFrame(index=tr_idx, columns=Y_B_MERKMALE, dtype=float)
                best_params_je_y = []
                predict_zeit_summe = 0
                for y_col in Y_B_MERKMALE:
                    grid = GridSearchCV(clone(konfig["modell"]), konfig["params"], scoring="r2", cv=innere_cv, n_jobs=-1)
                    grid.fit(X_tr, y_tr[y_col])
                    t1 = time.time()
                    y_pred_val[y_col] = grid.predict(X_val)
                    y_pred_train[y_col] = grid.predict(X_tr)
                    predict_zeit_summe += time.time() - t1
                    best_params_je_y.append(grid.best_params_)
                fit_zeit = time.time() - t0 - predict_zeit_summe
                predict_zeit = predict_zeit_summe
                fold_best_params.append(best_params_je_y)

            fold_fit_zeiten.append(fit_zeit)
            fold_predict_zeiten.append(predict_zeit)

            val_metriken_je_y = {y: berechne_metriken_b(y_val[y], y_pred_val[y], y_std_je_merkmal[y]) for y in Y_B_MERKMALE}
            train_metriken_je_y = {y: berechne_metriken_b(y_tr[y], y_pred_train[y], y_std_je_merkmal[y]) for y in Y_B_MERKMALE}
            fold_val_metriken.append(val_metriken_je_y)
            fold_train_metriken.append(train_metriken_je_y)

        for metrik_key in ["mae", "rmse", "r2", "median_ae", "rmse_normiert", "mae_normiert"]:
            val_werte = [np.mean([fm[y][metrik_key] for y in Y_B_MERKMALE]) for fm in fold_val_metriken]
            train_werte = [np.mean([fm[y][metrik_key] for y in Y_B_MERKMALE]) for fm in fold_train_metriken]
            eintrag[f"{metrik_key}_mean"] = np.mean(val_werte)
            eintrag[f"{metrik_key}_std"] = np.std(val_werte)
            eintrag[f"{metrik_key}_gap"] = np.mean(train_werte) - np.mean(val_werte)

        eintrag["fit_zeit_sekunden_mean"] = np.mean(fold_fit_zeiten)
        eintrag["predict_zeit_sekunden_mean"] = np.mean(fold_predict_zeiten)
        eintrag["best_params_je_fold"] = str(fold_best_params)

    except Exception as e:
        eintrag["status"] = "fehlgeschlagen"
        eintrag["fehler"] = f"{type(e).__name__}: {str(e)[:300]}"

    tuning_ergebnisse_b.append(eintrag)
    try:
        pd.DataFrame(tuning_ergebnisse_b).to_csv(OUTPUT_CSV_TUNING_B, index=False)
        speicher_status = "gespeichert"
    except Exception as speicher_fehler:
        speicher_status = f"SPEICHERN FEHLGESCHLAGEN: {speicher_fehler}"

    if eintrag["status"] == "ok":
        print(f"[{modell_idx+1}/{len(SUCHRAEUME_B)}] {modell_name:22s} -> ok "
              f"(R2={eintrag['r2_mean']:.4f}, Gap={eintrag['r2_gap']:.4f}, "
              f"FitZeit={eintrag['fit_zeit_sekunden_mean']:.2f}s, PredictZeit={eintrag['predict_zeit_sekunden_mean']:.4f}s) ({speicher_status})")
    else:
        print(f"[{modell_idx+1}/{len(SUCHRAEUME_B)}] {modell_name:22s} -> FEHLGESCHLAGEN ({speicher_status})")

print(f"\nGesamtzeit: {time.time()-start_gesamt:.1f}s")
print(f"Gespeichert in: {OUTPUT_CSV_TUNING_B}")

[1/8] Ridge                  -> ok (R2=0.5675, Gap=0.0062, Zeit=1.38s)
[2/8] kNN                    -> ok (R2=0.4918, Gap=0.0473, Zeit=0.10s)
[3/8] RandomForest           -> ok (R2=0.5464, Gap=0.1074, Zeit=32.92s)
[4/8] MLP                    -> ok (R2=0.5359, Gap=0.0249, Zeit=13.99s)
[5/8] SVR                    -> ok (R2=0.5632, Gap=0.0136, Zeit=8.72s)
[6/8] HistGradientBoosting   -> ok (R2=0.5503, Gap=0.0933, Zeit=7.41s)
[7/8] XGBoost                -> ok (R2=0.5246, Gap=0.2106, Zeit=9.52s)
[8/8] LightGBM               -> ok (R2=0.5502, Gap=0.0976, Zeit=37.52s)

Gesamtzeit: 559.8s
